# 06 - Conversational Interface
## Module D: Natural Language Query Interface

Implements intent classification, filtering, and templated response generation so users can query the corpus in plain English, e.g. *'Show me positive tech news from this week'*.> **Setup note:** This notebook uses the `src/` package from the project root.
> If running in Google Colab, first mount/clone the repo so `src/` is on the path,
> and place the Kaggle `BBC News Train.csv` in `data/raw/` for the real dataset
> (otherwise the bundled offline sample in `data/sample/` is used automatically).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.data_processing.data_loader import load_news_data, dataset_source
print("Dataset source:", dataset_source())


Dataset source: Bundled sample dataset (demo only): /home/claude/ITAI2373-NewsBot-Final/data/sample/sample_news.csv


In [2]:
df = pd.read_csv('../data/processed/articles_processed.csv') if os.path.exists('../data/processed/articles_processed.csv') else load_news_data()
from src.analysis.sentiment_analyzer import SentimentAnalyzer
if 'sentiment_label' not in df.columns:
    df = SentimentAnalyzer().analyze_dataframe(df, text_col='content')
df.shape

(200, 15)

## Intent classification

In [3]:
from src.conversation.intent_classifier import IntentClassifier

ic = IntentClassifier()
for q in ["Show me positive tech news from this week",
          "how many negative sport articles are there?",
          "summarize the business news",
          "articles about elections"]:
    print(q, "->", ic.classify(q))

Show me positive tech news from this week -> {'intent': 'filter_by_sentiment_category', 'all_matched_intents': ['filter_by_category', 'filter_by_sentiment', 'filter_by_time'], 'entities': {'category': 'tech', 'sentiment': 'positive', 'time_window': '7D', 'time_phrase': 'this week'}}
how many negative sport articles are there? -> {'intent': 'filter_by_sentiment_category', 'all_matched_intents': ['filter_by_category', 'filter_by_sentiment', 'count'], 'entities': {'category': 'sport', 'sentiment': 'negative'}}
summarize the business news -> {'intent': 'filter_by_category', 'all_matched_intents': ['filter_by_category', 'summarize'], 'entities': {'category': 'business'}}
articles about elections -> {'intent': 'search', 'all_matched_intents': ['search'], 'entities': {}}


## Full conversational query pipeline (with follow-up context)

In [4]:
from src.conversation.query_processor import QueryProcessor
from src.conversation.response_generator import ResponseGenerator

qp = QueryProcessor(df, text_col='content')
rg = ResponseGenerator()

turn1 = qp.process("Show me positive tech news")
print("USER: Show me positive tech news")
print("BOT: ", rg.format_response(turn1)['message'])

turn2 = qp.process("now show negative")   # follow-up reuses category=tech from context
print("\nUSER: now show negative")
print("BOT: ", rg.format_response(turn2)['message'])

USER: Show me positive tech news
BOT:  I found 33 articles matching your filter. Here are the top results. (category = tech; sentiment = positive)

USER: now show negative
BOT:  Here are 6 results for your query. (category = tech; sentiment = negative)


## Semantic (free-text) search for queries outside the structured filters

In [5]:
hits = qp.semantic_search("interest rate policy decisions", top_k=3)
for h in hits:
    print(f"[{h['score']:.3f}] {h['article']['title']}")

[0.275] Polling data released this week suggests public opinion
[0.270] The prime minister addressed concerns over immigration policy
[0.269] Polling data released this week suggests public opinion


## Key Takeaways
- `IntentClassifier` is a lightweight, explainable, fully offline rule-based classifier - appropriate given the fixed, small set of supported intents (vs. training a full intent model that would need thousands of labeled examples).
- `QueryProcessor` maintains simple conversational context so follow-up queries reuse prior filters.
- Structured filters (category/sentiment/time) and semantic search are complementary: structured filters answer precise questions, semantic search handles open-ended topical queries.